In [ ]:
import os, sys, json, shutil, numpy as np, subprocess, time
from pathlib import Path

COMP = Path('/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation')
WORK = Path('/kaggle/working')
sys.path.insert(0, str(COMP / 'kickmix'))
sys.path.insert(0, str(COMP / 'verifiers'))
sys.path.insert(0, '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit')
sys.path.insert(0, '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit/verifier')

SRC = Path('/kaggle/input/datasets/michaelrustad/vqcc-dfthc-r10-merged-groups/src')
sys.path.insert(0, str(SRC))

import types
dfthcsos = types.ModuleType('dfthcsos')
sys.modules['dfthcsos'] = dfthcsos
scoreinfra = types.ModuleType('dfthcsos.scoreinfra')
import dfthc_math as dm
for attr in dir(dm):
    if not attr.startswith('__'):
        setattr(scoreinfra, attr, getattr(dm, attr))
sys.modules['dfthcsos.scoreinfra'] = scoreinfra
dfthcsos.scoreinfra = scoreinfra
print('Shim loaded')

bp_members = Path('/kaggle/input/datasets/busyaprime/33-qcc-v8-members')
sub_dir = WORK / 'submission'
sub_dir.mkdir(exist_ok=True)
(sub_dir / 'dfthcsos').mkdir(exist_ok=True)

shutil.copy2(bp_members / 'invsqrt.kmb', sub_dir / 'invsqrt.kmb')
shutil.copy2(bp_members / 'dqi.kmb', sub_dir / 'dqi.kmb')
for p in bp_members.glob('dfthcsos__*.kmb'):
    shutil.copy2(p, sub_dir / 'dfthcsos' / p.name.replace('dfthcsos__', ''))
shutil.copy2(Path('/kaggle/input/datasets/michaelrustad/vqcc-dfthc-r10-merged-groups/fact/dfthc_r10_b129_c46_brot8.npz'), sub_dir / 'dfthcsos' / 'submission.npz')

print('Setup complete')


In [ ]:
IN = sub_dir / 'dfthcsos' / 'submission.npz'
RUNS = WORK / 'runs'
RUNS.mkdir(exist_ok=True)
TAG = 'bp_opt'
CAP = '0.0193'

print('Step 1: finish.py')
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(SRC / 'finish.py'), str(IN), str(RUNS / f'{TAG}_fin.npz'), '--kappa', '0'],
    capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=str(SRC) + os.pathsep + str(COMP / 'kickmix') + os.pathsep + str(COMP / 'verifiers') + os.pathsep + '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit' + os.pathsep + '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit/verifier')
)
print(result.stdout[-500:] if result.stdout else 'NO STDOUT')
print(result.stderr[-500:] if result.stderr else 'NO STDERR')
print(f'  Time: {time.time()-t0:.0f}s')
print('Output exists:', (RUNS / f'{TAG}_fin.npz').exists())


In [ ]:
print('Step 2: gauge.py')
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(SRC / 'gauge.py'), str(RUNS / f'{TAG}_fin.npz'), str(RUNS / f'{TAG}_g.npz'), '--sweeps', '60'],
    capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=str(SRC) + os.pathsep + str(COMP / 'kickmix') + os.pathsep + str(COMP / 'verifiers') + os.pathsep + '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit' + os.pathsep + '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit/verifier')
)
print(result.stdout[-500:] if result.stdout else 'NO STDOUT')
print(result.stderr[-500:] if result.stderr else 'NO STDERR')
print(f'  Time: {time.time()-t0:.0f}s')
print('Output exists:', (RUNS / f'{TAG}_g.npz').exists())


In [ ]:
print('Step 3: lamopt.py')
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(SRC / 'lamopt.py'), str(RUNS / f'{TAG}_g.npz'), str(RUNS / f'{TAG}_lam.npz'),
     '--steps', '1500', '--lr-w', '5e-7', '--lr-o', '2e-6', '--log', '500', '--cap', CAP],
    capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=str(SRC) + os.pathsep + str(COMP / 'kickmix') + os.pathsep + str(COMP / 'verifiers') + os.pathsep + '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit' + os.pathsep + '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit/verifier')
)
print(result.stdout[-1000:] if result.stdout else 'NO STDOUT')
print(result.stderr[-1000:] if result.stderr else 'NO STDERR')
print(f'  Time: {time.time()-t0:.0f}s')
print('Output exists:', (RUNS / f'{TAG}_lam.npz').exists())


In [ ]:
print('Step 4: finish.py again')
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(SRC / 'finish.py'), str(RUNS / f'{TAG}_lam.npz'), str(RUNS / f'{TAG}_lamfin.npz'), '--kappa', '0'],
    capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=str(SRC) + os.pathsep + str(COMP / 'kickmix') + os.pathsep + str(COMP / 'verifiers') + os.pathsep + '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit' + os.pathsep + '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit/verifier')
)
print(result.stdout[-500:] if result.stdout else 'NO STDOUT')
print(result.stderr[-500:] if result.stderr else 'NO STDERR')
print(f'  Time: {time.time()-t0:.0f}s')
print('Output exists:', (RUNS / f'{TAG}_lamfin.npz').exists())


In [ ]:
print('Step 5: gauge.py again')
t0 = time.time()
result = subprocess.run(
    [sys.executable, str(SRC / 'gauge.py'), str(RUNS / f'{TAG}_lamfin.npz'), str(RUNS / f'{TAG}_final.npz'), '--sweeps', '60'],
    capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=str(SRC) + os.pathsep + str(COMP / 'kickmix') + os.pathsep + str(COMP / 'verifiers') + os.pathsep + '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit' + os.pathsep + '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit/verifier')
)
print(result.stdout[-500:] if result.stdout else 'NO STDOUT')
print(result.stderr[-500:] if result.stderr else 'NO STDERR')
print(f'  Time: {time.time()-t0:.0f}s')
print('Output exists:', (RUNS / f'{TAG}_final.npz').exists())


In [ ]:
import sys
sys.path.insert(0, '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit')
sys.path.insert(0, '/kaggle/input/datasets/evanpieser/qcompile-verifiable/starter_kit/verifier')
import importlib.util
spec = importlib.util.spec_from_file_location('dfthcsos_verify', str(COMP / 'verifiers' / 'dfthcsos' / 'verify.py'))
dfthcsos_verify = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dfthcsos_verify)
CV = dfthcsos_verify.verify
from starter.task3_circuits.build_circuits import build_submission

final_npz = RUNS / f'{TAG}_final.npz'
if not final_npz.exists():
    print('ERROR: final.npz not found, trying lamfin')
    final_npz = RUNS / f'{TAG}_lamfin.npz'
if not final_npz.exists():
    print('ERROR: lamfin.npz not found, trying lam')
    final_npz = RUNS / f'{TAG}_lam.npz'
if not final_npz.exists():
    print('ERROR: lam.npz not found, trying g')
    final_npz = RUNS / f'{TAG}_g.npz'
if not final_npz.exists():
    print('ERROR: g.npz not found, trying fin')
    final_npz = RUNS / f'{TAG}_fin.npz'

print('Using factorization:', final_npz)

shutil.copy2(final_npz, sub_dir / 'dfthcsos' / 'submission.npz')

result = build_submission(
    'FeMoCo76o113e', final_npz,
    tree='log_depth', brot=8, batch_size=5, rounding='nearest',
    compiler='qrom',
    output_directory=sub_dir / 'dfthcsos', overwrite=True,
)
print('Build result score:', result['score'])

r = CV(sub_dir / 'dfthcsos')
print('Full verify:', r['status'], r['score'])

zip_path = WORK / 'submission_optimized.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for p in sorted(sub_dir.rglob('*')):
        if p.is_file():
            z.write(p, str(p.relative_to(sub_dir)))
print('Wrote', zip_path, zip_path.stat().st_size, 'bytes')
